In [5]:
from player_stats import PlayerStats
from fetch_data import load_data, get_player_rows


In [4]:
import numpy as np
import pandas as pd

from scripts.fetch_data import clean_data

In [18]:
from collections import defaultdict
import numpy as np
from fetch_data import load_data, clean_data

COLS = ['ace', 'svpt', '1stIn', '1stWon', '2ndWon', 'bpFaced', 'bpSaved']

def get_features(s, surf_wins, surf_losses):
    """s holds one player's running totals. Raises ZeroDivisionError if data is missing."""
    second_pts = s['svpt'] - s['1stIn']
    return np.array([
        s['ace'] / s['svpt'] * 100,                                            # ace rate
        s['bpSaved'] / s['bpFaced'] * 100,                                     # bp saved
        (s['opp_bpFaced'] - s['opp_bpSaved']) / s['opp_bpFaced'] * 100,        # bp conversion
        surf_wins / (surf_wins + surf_losses) * 100,                           # surface win rate
        s['1stIn'] / s['svpt'] * 100,                                          # first serve %
        s['1stWon'] / s['1stIn'] * 100,                                        # first serve win %
        s['2ndWon'] / second_pts * 100,                                        # second serve win %
        (s['opp_svpt'] - s['opp_1stWon'] - s['opp_2ndWon']) / s['opp_svpt'] * 100,  # return pts won
        s['wins'] / (s['wins'] + s['losses']) * 100,                           # overall win %
    ])

def add_match(s, own, opp):
    for c in COLS:
        s[c] += own[c]
        s['opp_' + c] += opp[c]

matches = clean_data(load_data(from_year = 2000 )).reset_index(drop=True)
stat_cols = [p + c for p in ('w_', 'l_') for c in COLS]
matches[stat_cols] = matches[stat_cols].fillna(0)   # same as .sum() ignoring NaN
rows = matches.to_dict('records')

stats = defaultdict(lambda: defaultdict(float))     # player -> totals
surface_record = defaultdict(lambda: [0, 0])        # (player, surface) -> [wins, losses]

X_rows, Y_rows, counter = [], [], 0

for m in rows:
    p1, p2, surface = m['winner_name'], m['loser_name'], m['surface']

    try:
        w1, l1 = surface_record[(p1, surface)]
        w2, l2 = surface_record[(p2, surface)]
        v1 = get_features(stats[p1], w1, l1)
        v2 = get_features(stats[p2], w2, l2)
        X_rows.append(v1 - v2); Y_rows.append(1)
        X_rows.append(v2 - v1); Y_rows.append(0)
    except ZeroDivisionError:
        counter += 1

    # update AFTER using the match
    w_own = {c: m['w_' + c] for c in COLS}
    l_own = {c: m['l_' + c] for c in COLS}
    add_match(stats[p1], w_own, l_own)
    add_match(stats[p2], l_own, w_own)
    stats[p1]['wins'] += 1
    stats[p2]['losses'] += 1
    surface_record[(p1, surface)][0] += 1
    surface_record[(p2, surface)][1] += 1

X, Y = np.array(X_rows), np.array(Y_rows)



In [ ]:
X.shape, Y.shape

In [37]:
X

array([[  8.70397478, -30.95238095, -21.42857143, ...,  -1.26984127,
         -6.01228443,   0.        ],
       [ -8.70397478,  30.95238095,  21.42857143, ...,   1.26984127,
          6.01228443,   0.        ],
       [  9.22959573,  33.33333333, -19.64285714, ...,  -0.74074074,
         -6.26566416,   0.        ],
       ...,
       [-11.50307835,  -4.66240089,   1.93444065, ...,   3.66665515,
          3.41795079,  -9.30022943],
       [ -1.62725673,  -1.8838666 ,   6.16565573, ...,   0.20587784,
          1.47029736,   1.79430263],
       [  1.62725673,   1.8838666 ,  -6.16565573, ...,  -0.20587784,
         -1.47029736,  -1.79430263]], shape=(141054, 9))

In [20]:
split = int(len(X) * 0.8) // 2 * 2
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, Y_train)
print(f"accuracy: {model.score(X_test, Y_test)}")

accuracy: 0.6157663405642989


In [25]:
print(model[-1].coef_)

[[ 0.09326948  0.00180351 -0.03778812  0.23755181  0.11988731  0.30697035
   0.18671715  0.40788432  0.10218553]]


In [26]:
# ace rate
# bp saved
# bp conversion
# surface win rate
# first serve %
# first serve win %
# second serve win %
# return pts won
# overall win %

In [40]:
data = load_data(from_year = 2000)
data = clean_data(data)
data1 = get_player_rows(data, 'Rafael Nadal')
data2 = get_player_rows(data, 'Novak Djokovic')

player1 = PlayerStats(data1, 'Rafael Nadal')
player2 = PlayerStats(data2, 'Novak Djokovic')

p1_vect = player1.get_player_features('Clay')
p2_vect = player2.get_player_features('Clay')

new_sample = np.array(p1_vect - p2_vect)
new_sample = new_sample.reshape(1, -1)

pred = model.predict(new_sample)
prob = model.predict_proba(new_sample)

print(f"Prediction: {pred}")
print(f"Probability: {prob}")


Prediction: [1]
Probability: [[0.47632861 0.52367139]]
